In [3]:
import os
import pandas as pd
from IPython.display import display, Markdown
from tqdm.auto import tqdm # makes pretty progress bars
tqdm.pandas()
# this should be familiar!
from dotenv import load_dotenv
load_dotenv()

True

In [5]:
full_data = pd.read_csv("tyler_trade_buys_only.csv")
full_data.head()

,date,trade_type,side,outcome,amount,price,total,title,user
0,2026-07-04 16:28,TRADE,BUY,No,3411.91,0.360000,1228.2876,US-Iran 60 day negotiation period extended?,0xe4de861d59e174d3d153a5968bc122c66f30c949
1,2026-07-04 16:19,TRADE,BUY,No,426.13,0.639695,272.5932,Strait of Hormuz traffic returns to normal by ...,0xe4de861d59e174d3d153a5968bc122c66f30c949
2,2026-07-04 16:18,TRADE,BUY,No,5000.00,0.485876,2429.3786,Strait of Hormuz traffic returns to normal by ...,0xe4de861d59e174d3d153a5968bc122c66f30c949
3,2026-07-04 16:17,TRADE,BUY,No,15.88,0.630000,10.0044,Strait of Hormuz traffic returns to normal by ...,0xe4de861d59e174d3d153a5968bc122c66f30c949
4,2026-07-04 16:17,TRADE,BUY,No,1603.98,0.630000,1010.5074,Strait of Hormuz traffic returns to normal by ...,0xe4de861d59e174d3d153a5968bc122c66f30c949


In [6]:
SIZE_OF_SAMPLE_FOR_HANDCODING = 100
##
## take a random sample to test our prompts against
text_column_name = "title" # change this to match your own dataset
full_data.dropna(subset=[text_column_name], inplace=True)
full_data["date"] = pd.to_datetime(full_data["date"])

# if you already did the handcoding, make sure the sample you're using is the right set.
WE_ALREADY_HANDCODED = os.path.exists("handcoded.csv")
if WE_ALREADY_HANDCODED:
  data_sample = pd.read_csv("handcoded.csv")
else:
  data_sample = full_data.sample(n=SIZE_OF_SAMPLE_FOR_HANDCODING, random_state=613)

data_sample = data_sample.copy() # avoid annoying warnings!

In [79]:
data_sample.to_csv("sample.csv")
data_sample

,Column1,date,trade_type,side,outcome,amount,price,total,title,user,ai_prompt,ai_guess,groundtruth
0,3082,7/10/2026 8:39,TRADE,BUY,BetBoom Team,13.770832,0.520,7.160833,Counter-Strike: FaZe vs BetBoom Team - Map 1 W...,0x96f795d0821b75a1c7f886b2c1cd49d108b7d6ae,\nYou are an expert in text interpretation and...,gaming,gaming
1,2921,6/9/2026 23:16,TRADE,BUY,Yes,25.020000,0.950,23.769000,Will Pamela Evette win the first round of the ...,0x2b9dbf4b6e0e11309a9d6d2a09b72f65f652adc0,\nYou are an expert in text interpretation and...,elections_global_and_domestic,elections_global_and_domestic
2,5588,7/4/2026 10:00,TRADE,BUY,Under,2000.000000,0.290,580.000000,Canada vs. Morocco: O/U 1.5,0x760907c8840491c8c86fddb843d3f1d801786c4e,\nYou are an expert in text interpretation and...,sports,sports
3,19168,7/10/2026 15:11,TRADE,BUY,Yes,166.670000,0.970,161.669900,Nathan Ngoy: 1+ shots,0x2843d543c60a54106f572ab74f0e2e04f293b3d2,\nYou are an expert in text interpretation and...,sports,sports
4,15089,2/6/2026 8:17,TRADE,BUY,Yes,5.555554,0.640,3.555555,"Will Trump say ""Charlie Kirk"" during the State...",0xb1a2757fd88d6f66255f53565ada24d875b257fe,\nYou are an expert in text interpretation and...,political_issues_except_us_elections,political_issues_except_us_elections
...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,41257,6/27/2026 15:21,TRADE,BUY,Down,2.000000,0.500,1.000000,"Ethereum Up or Down - June 27, 3:20PM-3:25PM ET",0xcece2f5f75676c11ca0151a70507e045ca63a1e7,\nYou are an expert in text interpretation and...,crypto_currencies,crypto_currencies
96,13791,7/6/2026 16:47,TRADE,BUY,No,200.000000,0.670,134.000000,Will 60 ships transit the Strait of Hormuz on ...,0xeca0c0888e34df59589c67fbe53dc7f298b5e8f8,\nYou are an expert in text interpretation and...,geopolitics_except_elections,usa_iran_conflict
97,21106,6/25/2026 14:13,TRADE,BUY,Yes,18.000000,0.238,4.284000,Will Sweden win on 2026-06-25?,0xb8d8bc6acbe4b4596750b8ccd0f227eb7bd366a7,\nYou are an expert in text interpretation and...,sports,sports
98,31998,7/10/2026 15:12,TRADE,BUY,Yes,10.000000,0.520,5.200000,"Will Trump say ""Future President"" in July?",0x6b0ebbef2630b9a0da35594bf7ddd1911246e3ea,\nYou are an expert in text interpretation and...,political_issues_except_us_elections,political_issues_except_us_elections


In [7]:
## prompt
## start with the one YOU tried in the LLM web interface
## don't worry, we'll adjust it later.
## Jeremy's is at the very bottom, if you need to reference it, but please come
##   up with your own.


# Be sure to list the categories (defined below!) with the magic phrase {categories}
prompt_base = """
You are an expert in text interpretation and classification.
Please read this prediction market title and determine which category best describes it. 
You must choose from the strict list of categories defined below. Return ONLY the exact Category Code, nothing else.

Categories and Definitions:
- sports: Any professional or collegiate sports events, games, or athlete performance metrics.
- political_issues_except_us_elections: Broader domestic political events, legislative outcomes, or policy decisions in the United States, excluding direct election results.
- gaming: Video games, esports tournaments, or gaming industry news.
- usa_iran_conflict: Any geopolitical, military, or diplomatic events specifically involving the United States and Iran. Developments in the straight of Hormuz included.
- elections_global_and_domestic: Political elections, including polling, candidates, and winners, both within the US and internationally.
- geopolitics_except_elections: All politics outside the USA, except elections and wars. International relations, global treaties, or global diplomatic events, excluding elections and wars.
- war_events: Military conflicts (excluding US-Iran). Examples of other ongoing wars are Russia-Ukraine, Israel-Lebanon. Israel-Iran counts as USA-Iran.
- tech: Technology companies, artificial intelligence, product launches, or science and space exploration.
- crypto_currencies: Blockchain technology, cryptocurrency prices, NFT markets, or decentralized finance (DeFi).
- pop_culture_and_entertainment: Celebrities, music, movies, awards shows, social media influencers, or viral internet events.
- crime: Criminal trials, verdicts, arrests, or legal proceedings.
- other: if it can not be defined by any other category. Resort to this only if necessary.

Here is the exact list of Category Codes to output: {categories}

Here is the market title: {market_title}
Do a really good job and make me proud. I will kick your grandma in the belly if you don't.

"""

from enum import Enum, IntEnum
# defining the categories "in a special way that will make the model follow directions"
# force the AI to ONLY guess fromn the options we ask you.
# you can change these names! (and perhaps should!)
# - the AI sees the _value_, not the key (it sees "broader_political_issues_except_us_election", not "other_politics")
# - the key is just for you -- a shorthand
# - some AIs have trouble with complicated names, so it's better to stick to lowercase, no special characters
#     (as of early June 2026 -- maybe this will be fixed)
# name them what they are because the llm needs to read them
class MarketCategoryOptions(str, Enum):
    sports = "sports"
    other_us_politics = "political_issues_except_us_elections"
    gaming = "gaming"
    us_iran_conflict = "usa_iran_conflict"
    elections = "elections_global_and_domestic"
    geopolitics = "geopolitics_except_elections"
    war = "war_events"
    tech = "tech"
    crypto = "crypto_currencies"
    pop_culture = "pop_culture_and_entertainment"
    crime = "crime"
    other = "other"


# for convenience, make a column in our dataframes with the prompt for each item
data_sample["ai_prompt"] = data_sample.apply(lambda row: prompt_base.format(
    market_title=row["title"],
    categories=", ".join(['"{}"'.format(opt.value) for opt in MarketCategoryOptions])
), axis="columns")
full_data["ai_prompt"] = full_data.apply(lambda row: prompt_base.format(
    market_title=row["title"],
    categories=", ".join(['"{}"'.format(opt.value) for opt in MarketCategoryOptions])
), axis="columns")

In [8]:
USE_OPENROUTER=True

from anthropic import Anthropic

if USE_OPENROUTER:
  from openrouter import OpenRouter
else:
  from openai import OpenAI
  from mistralai.client import Mistral

if USE_OPENROUTER:
  openrouter_client = OpenRouter(api_key=os.environ.get("OPENROUTER_API_KEY"))
else:
  openai_client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY_LEDE"))
  mistral_client = Mistral(api_key=os.environ.get("MISTRAL_API_KEY_LEDE"))
anthropic_client = Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY_LEDE"),)

In [9]:
data_sample["ai_prompt"].iloc[0]

'\nYou are an expert in text interpretation and classification.\nPlease read this prediction market title and determine which category best describes it. \nYou must choose from the strict list of categories defined below. Return ONLY the exact Category Code, nothing else.\n\nCategories and Definitions:\n- sports: Any professional or collegiate sports events, games, or athlete performance metrics.\n- political_issues_except_us_elections: Broader domestic political events, legislative outcomes, or policy decisions in the United States, excluding direct election results.\n- gaming: Video games, esports tournaments, or gaming industry news.\n- usa_iran_conflict: Any geopolitical, military, or diplomatic events specifically involving the United States and Iran. Developments in the straight of Hormuz included.\n- elections_global_and_domestic: Political elections, including polling, candidates, and winners, both within the US and internationally.\n- geopolitics_except_elections: All politics

In [10]:
from strip_tags import strip_tags
import tiktoken
def count_tokens(model, text):
  if "claude" in model and os.environ.get("ANTHROPIC_API_KEY_LEDE"):
    return anthropic_client.messages.count_tokens(model=model, messages=[{"content": text, "role": "user"}])
  elif "gpt" in model:
    encoding = tiktoken.encoding_for_model('gpt-4o') # most modern models use the same tokenizer
    tokens = encoding.encode(text)
    return len(tokens)
  else:
    return count_tokens("gpt-4o", text) # for estimation purposes, pretend mistral is chatgpt...

INPUT_TOKEN_COSTS = {
    "gpt-5-mini": 0.40 / 1_000_000,  # https://openai.com/api/pricing/
    "gpt-5.4-mini": 0.75 / 1_000_000,
    "gpt-5.4": 2.5 / 1_000_000,
    "gpt-5.5": 5 / 1_000_000,
    "gpt-5.4-nano": 0.2 / 1_000_000,
    "mistral-medium-latest": 1.5 / 1_000_000, # https://mistral.ai/pricing#api-pricing
    "mistral-large-latest":  0.5 / 1_000_000,  # No, I don't know why medium is more expensive than large.
    "claude-haiku-4.5": 1 / 1_000_000,
    "claude-sonnet-4.6": 3 / 1_000_000
}
# ignores outputs, since the model will output fairly little text

def estimate_cost(model, token_count):
    return token_count * INPUT_TOKEN_COSTS[model]

In [11]:
DEFAULT_MODEL_TO_USE = 'gpt-5.4-nano'
display(Markdown("Pick a model to use: \n" + "\n - ".join(list(INPUT_TOKEN_COSTS.keys()))))
MODEL_TO_USE = None

while MODEL_TO_USE not in INPUT_TOKEN_COSTS.keys():
  MODEL_TO_USE = input(f"type a model name (press enter for default: {DEFAULT_MODEL_TO_USE}): ").strip()
  if MODEL_TO_USE == "":
    MODEL_TO_USE = DEFAULT_MODEL_TO_USE

Pick a model to use: 
gpt-5-mini
 - gpt-5.4-mini
 - gpt-5.4
 - gpt-5.5
 - gpt-5.4-nano
 - mistral-medium-latest
 - mistral-large-latest
 - claude-haiku-4.5
 - claude-sonnet-4.6

type a model name (press enter for default: gpt-5.4-nano):  claude-haiku-4.5


In [13]:
## cost estimation sample

count_tokens_for_our_model = lambda text: count_tokens(MODEL_TO_USE, text)

token_count_sample = count_tokens_for_our_model("SAMPLE RESPONSE SAMPLE RESPONSE".join(data_sample["ai_prompt"]))
"Sample would cost: ${:.2f}".format(estimate_cost(MODEL_TO_USE, token_count_sample))

'Sample would cost: $0.05'

In [14]:
## cost estimation for full dataset
token_count_full = count_tokens_for_our_model("SAMPLE RESPONSE SAMPLE RESPONSE".join(full_data["ai_prompt"]))
"Full dataset would cost: ${:.2f}".format(estimate_cost(MODEL_TO_USE, token_count_full))

'Full dataset would cost: $19.50'

In [15]:
## a lot of boilerplate for sending our prompt + tweet to the model and getting an answer
from pydantic import BaseModel


class MarketCategoryValidOptions(BaseModel):
  classification: MarketCategoryOptions

SYSTEM_PROMPT = "You are a helpful assistant"

def anthropic_classify(prompt_including_title):
    # put our prompt into the blob that OpenAI expects
    message = anthropic_client.messages.parse(
        max_tokens=1024,
        # system=SYSTEM_PROMPT, # causes trouble with structured outputs, oddly
        messages = [
            {
                "role": "user",
                "content": prompt_including_title,
            }
        ],
        model=MODEL_TO_USE,
        output_format=MarketCategoryOptions
    )
    return message.content[0].parsed_output.classification.value if message.content[0].parsed_output else None

def mistral_classify(prompt_including_title):
    # put our prompt into the blob that OpenAI expects
    chat_response = mistral_client.chat.parse(
      model=MODEL_TO_USE,
      messages=[
          {
              "role": "system",
              "content": SYSTEM_PROMPT
          },
          {
              "role": "user",
              "content": prompt_including_title
          },
      ],
      response_format=MarketCategoryOptions,
      max_tokens=256,
      temperature=0
    )
    return chat_response.choices[0].message.parsed.classification.value if chat_response.choices[0].message.parsed else None

def chatgpt_classify(prompt_including_title):
    # put our prompt into the blob that OpenAI expects
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": prompt_including_title,
        }
    ]
    chat_completion = openai_client.responses.parse(
        input=messages,
        model=MODEL_TO_USE,
        text_format=MarketCategoryOptions,
    )

    # get the answer out of the blob that OpenAI returns.
    resp = chat_completion.output_parsed.classification.value if chat_completion.output_parsed else None
    return resp

openrouter_model_names = {
    "gpt-5-mini": "openai/gpt-5-mini",
    "gpt-5.4-mini": "openai/gpt-5.4-mini",
    "gpt-5.4": "openai/gpt-5.4",
    "gpt-5.5": "openai/gpt-5.5",
    "gpt-5.4-nano": "openai/gpt-5.4-nano",
    "mistral-medium-latest": "mistralai/mistral-medium-3-5",
    "mistral-large-latest": "mistralai/mistral-large-2512",
    "claude-haiku-4.5": "anthropic/claude-haiku-4.5",
    "claude-sonnet-4.6": "anthropic/claude-4.6-sonnet"
}
def openrouter_classify(prompt_including_title):
    response = openrouter_client.chat.send(
        model=openrouter_model_names[MODEL_TO_USE],
        messages=[
            {"role": "user", "content": prompt_including_title}
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "musk_tweet_classification",
                "strict": True,
                "schema": MarketCategoryValidOptions.model_json_schema(),
            },
        },
    )
    return  MarketCategoryValidOptions.model_validate_json(
        response.choices[0].message.content
    ).classification.value

def classify(prompt_including_title):
  "A wrapper for the provider-specific classification functions"
  if USE_OPENROUTER:
    return openrouter_classify(prompt_including_title)
  else:
    if "mistral" in MODEL_TO_USE == "mistral-medium-latest":
      return mistral_classify(prompt_including_title)
    elif "claude" in MODEL_TO_USE:
      return anthropic_classify(prompt_including_title)
    elif "gpt" in MODEL_TO_USE:
      return chatgpt_classify(prompt_including_title)
    else:
      raise ValueError("Unknown model")

In [102]:
# ask ChatGPT about each and every tweet IN THE SAMPLE, using the prompt we made above
data_sample["ai_guess"] = data_sample["ai_prompt"].progress_apply(classify)
display(Markdown("## Here are a few results. How good did we do?"))
with pd.option_context("display.max_colwidth", 500):
  display(
      data_sample[[text_column_name, "ai_guess"]].head(5)
  )

100%|██████████████████████████████████████████████████████████████████████████████████████| 100/100 [01:50<00:00,  1.10s/it]


## Here are a few results. How good did we do?

,title,ai_guess
0,Counter-Strike: FaZe vs BetBoom Team - Map 1 Winner,gaming
1,Will Pamela Evette win the first round of the South Carolina Republican Governor Primary by less than 5%?,elections_global_and_domestic
2,Canada vs. Morocco: O/U 1.5,sports
3,Nathan Ngoy: 1+ shots,sports
4,"Will Trump say ""Charlie Kirk"" during the State of the Union address?",political_issues_except_us_elections


In [103]:
data_sample.head(10)

,Column1,date,trade_type,side,outcome,amount,price,total,title,user,ai_prompt,ai_guess,groundtruth
0,3082,7/10/2026 8:39,TRADE,BUY,BetBoom Team,13.770832,0.520,7.160833,Counter-Strike: FaZe vs BetBoom Team - Map 1 W...,0x96f795d0821b75a1c7f886b2c1cd49d108b7d6ae,\nYou are an expert in text interpretation and...,gaming,gaming
1,2921,6/9/2026 23:16,TRADE,BUY,Yes,25.020000,0.950,23.769000,Will Pamela Evette win the first round of the ...,0x2b9dbf4b6e0e11309a9d6d2a09b72f65f652adc0,\nYou are an expert in text interpretation and...,elections_global_and_domestic,elections_global_and_domestic
2,5588,7/4/2026 10:00,TRADE,BUY,Under,2000.000000,0.290,580.000000,Canada vs. Morocco: O/U 1.5,0x760907c8840491c8c86fddb843d3f1d801786c4e,\nYou are an expert in text interpretation and...,sports,sports
3,19168,7/10/2026 15:11,TRADE,BUY,Yes,166.670000,0.970,161.669900,Nathan Ngoy: 1+ shots,0x2843d543c60a54106f572ab74f0e2e04f293b3d2,\nYou are an expert in text interpretation and...,sports,sports
4,15089,2/6/2026 8:17,TRADE,BUY,Yes,5.555554,0.640,3.555555,"Will Trump say ""Charlie Kirk"" during the State...",0xb1a2757fd88d6f66255f53565ada24d875b257fe,\nYou are an expert in text interpretation and...,political_issues_except_us_elections,political_issues_except_us_elections
5,18625,7/9/2026 18:33,TRADE,BUY,Yes,5.140000,0.538,2.765320,Will Israel strike 4 countries in 2026?,0xa66790e2b0075e73c3781ddaf1d0b875a3aaceb3,\nYou are an expert in text interpretation and...,geopolitics_except_elections,war_events
6,17814,6/25/2026 16:26,TRADE,BUY,Yes,24.294116,0.170,4.129999,Will Sonia Backès be the next President of the...,0xc24fa1e0fda1552c830022807abd855aadbe220b,\nYou are an expert in text interpretation and...,elections_global_and_domestic,elections_global_and_domestic
7,35498,7/9/2026 11:19,TRADE,BUY,No,81.000000,0.885,71.685000,Will Elon Musk post 140-159 tweets from July 3...,0x6b17d237e6b7f74ce8171d6dc68e07bc21c803b2,\nYou are an expert in text interpretation and...,pop_culture_and_entertainment,pop_culture_and_entertainment
8,33775,7/10/2026 6:01,TRADE,BUY,No,10.000000,0.001,0.010000,Will OpenAI release a new frontier model on or...,0x66a1db98713015330a82f08273b682bee77e80d9,\nYou are an expert in text interpretation and...,tech,tech
9,13622,7/8/2026 17:25,TRADE,BUY,No,100.000000,0.260,26.000000,Will Graham Platner drop out by July 8?,0xeca0c0888e34df59589c67fbe53dc7f298b5e8f8,\nYou are an expert in text interpretation and...,elections_global_and_domestic,elections_global_and_domestic


In [104]:
#data_sample.to_csv("trades_for_hand_coding.csv") <-- we don't want to run this again (only 1st time)
data_sample.to_csv("handcoded.csv") #instead we overwrite the handcoded file

In [105]:
handcoded = pd.read_csv("handcoded.csv").set_index("title") # for some countries add `, delimiter=";"`
if "groundtruth" not in handcoded.columns:
  print("uh oh, your handcoded.csv doesn't have a column named groundtruth")
  assert "groundtruth" in handcoded.columns
if handcoded.groundtruth.isna().any() or (handcoded.groundtruth == '').any():
  print("uh oh, there are some blanks in your handcoded.csv's groundtruth column. go fix that!")

# if the ai_guess column got lost in the process... we'll grab it again
if "ai_guess" not in handcoded:
  handcoded = handcoded.merge(data_sample[["ai_guess"]], right_index=True, left_index=True)

In [106]:
from sklearn.metrics import accuracy_score
display(Markdown("Accuracy score: {:.1%}. Is that good?".format(
    accuracy_score(handcoded["groundtruth"], handcoded["ai_guess"])
)))


Accuracy score: 95.0%. Is that good?

In [107]:
# since running the AI over the whole dataset might cost a real amount of money
# this is just an extra check to make sure you want to do it.
# just hit enter (or type anything) to continue
# hit stop (to the left of this cell) to stop
okay_to_do_the_whole_thing = input()

In [ ]:
import time

def safe_classify(prompt_text):
    """A bodyguard function that retries the API if it fails, preventing crashes."""
    max_retries = 3
    for attempt in range(max_retries):
        try:
            # Try to run your original classify function
            return classify(prompt_text)
        except Exception as e:
            if attempt < max_retries - 1:
                # If it fails, wait a little bit (1s, 2s) and try again
                time.sleep(2 ** attempt) 
            else:
                # If it fails 3 times, don't crash! Just log an error in the CSV cell.
                # You can filter for these later to rerun them.
                return "API_ERROR"
                
# 1. Prepare your save file
output_filename = "ai_classified_tyler_trades_full.csv"
CHUNK_SIZE = 100

print(f"Starting classification for {len(full_data)} rows...")

# 2. Loop through the data in chunks of 100
for i in range(0, len(full_data), CHUNK_SIZE):
    
    # Slice out the current chunk
    chunk = full_data.iloc[i : i + CHUNK_SIZE].copy()
    
    # Run the AI on just these 100 rows
    print(f"\nProcessing rows {i} to {i + len(chunk)}...")
    chunk["ai_guess"] = chunk["ai_prompt"].progress_apply(safe_classify)
    
    # 3. Save the chunk to the CSV immediately
    # If it's the first chunk, write the headers. If it's a later chunk, just append to the bottom.
    if i == 0:
        chunk.to_csv(output_filename, index=False)
    else:
        chunk.to_csv(output_filename, index=False, mode='a', header=False)
        
    print(f"Saved rows {i} to {i + len(chunk)} to {output_filename}!")
    
    # Brief 2-second pause to let the API breathe and avoid rate limits
    time.sleep(2)

print("\n🎉 All 40,000 rows finished and securely saved!")

#BELOW IS THE CODE THAT JEREMY USED
#full_data["ai_guess"] = full_data["ai_prompt"].progress_apply(classify)
#full_data.to_csv("ai_classified_tyler_trades_full.csv", index=False)

Starting classification for 41291 rows...

Processing rows 0 to 100...


100%|██████████████████████████████████████████████████████████████████████████████████████| 100/100 [02:49<00:00,  1.69s/it]


Saved rows 0 to 100 to ai_classified_tyler_trades_full.csv!

Processing rows 100 to 200...


100%|██████████████████████████████████████████████████████████████████████████████████████| 100/100 [01:45<00:00,  1.06s/it]


Saved rows 100 to 200 to ai_classified_tyler_trades_full.csv!

Processing rows 200 to 300...


...

100%|██████████████████████████████████████████████████████████████████████████████████████| 100/100 [01:49<00:00,  1.10s/it]


Saved rows 24600 to 24700 to ai_classified_tyler_trades_full.csv!

Processing rows 24700 to 24800...


100%|██████████████████████████████████████████████████████████████████████████████████████| 100/100 [01:49<00:00,  1.09s/it]


Saved rows 24700 to 24800 to ai_classified_tyler_trades_full.csv!

Processing rows 24800 to 24900...


 99%|██████████████████████████████████████████████████████████████████████████████████████▏| 99/100 [02:13<00:04,  4.48s/it]

In [18]:
import time
check_df = pd.read_csv("ai_classified_tyler_trades_full.csv")
print(f"Safely saved rows on disk: {len(check_df)}")

START_ROW = len(check_df)  # Tell the script to skip the finished rows
output_filename = "ai_classified_tyler_trades_full.csv"
CHUNK_SIZE = 100

print(f"Resuming classification from row {START_ROW} out of {len(full_data)}...")

def safe_classify(prompt_text):
    """A bodyguard function that retries the API if it fails, preventing crashes."""
    max_retries = 3
    for attempt in range(max_retries):
        try:
            # Try to run your original classify function
            return classify(prompt_text)
        except Exception as e:
            if attempt < max_retries - 1:
                # If it fails, wait a little bit (1s, 2s) and try again
                time.sleep(2 ** attempt) 
            else:
                # If it fails 3 times, don't crash! Just log an error in the CSV cell.
                # You can filter for these later to rerun them.
                return "API_ERROR"

# ==========================================
# THE RESUME LOOP
# ==========================================
for i in range(START_ROW, len(full_data), CHUNK_SIZE):
    
    # Slice out the chunk starting from our resume point
    chunk = full_data.iloc[i : i + CHUNK_SIZE].copy()
    
    print(f"\nProcessing rows {i} to {i + len(chunk)}...")
    
    # Run the bodyguard function
    chunk["ai_guess"] = chunk["ai_prompt"].progress_apply(safe_classify)
    
    # FIX: Because we are resuming, we ALWAYS append ('a') and NEVER write headers
    chunk.to_csv(output_filename, index=False, mode='a', header=False)
    
    print(f"Saved rows {i} to {i + len(chunk)} to {output_filename}!")
    
    # Let the API breathe
    time.sleep(2)

print("\n🎉 All 40,000 rows finished and securely saved!")

Safely saved rows on disk: 24900
Resuming classification from row 24900 out of 41291...

Processing rows 24900 to 25000...


100%|██████████████████████████████████████████████████████████████████████████████████████| 100/100 [01:52<00:00,  1.12s/it]


Saved rows 24900 to 25000 to ai_classified_tyler_trades_full.csv!

Processing rows 25000 to 25100...


100%|██████████████████████████████████████████████████████████████████████████████████████| 100/100 [01:47<00:00,  1.07s/it]


Saved rows 25000 to 25100 to ai_classified_tyler_trades_full.csv!

Processing rows 25100 to 25200...


...

Saved rows 35000 to 35100 to ai_classified_tyler_trades_full.csv!

Processing rows 35100 to 35200...


100%|██████████████████████████████████████████████████████████████████████████████████████| 100/100 [05:55<00:00,  3.55s/it]


Saved rows 35100 to 35200 to ai_classified_tyler_trades_full.csv!

Processing rows 35200 to 35300...


  2%|█▊                                                                                      | 2/100 [00:07<06:27,  3.96s/it]


KeyboardInterrupt: 

In [19]:
df_saved = pd.read_csv("ai_classified_tyler_trades_full.csv")

In [20]:
print(f"Rows successfully analyzed: {len(df_saved)}")

Rows successfully analyzed: 35200
